In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import gc
import pickle
import re
import warnings

import numpy as np
import pandas as pd
import torch
from joblib import Parallel, delayed
from peft import PeftModel
from PIL import Image
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from tqdm.auto import tqdm
from transformers import AutoModel, AutoModelForImageTextToText, AutoProcessor

import os
os.environ['CUDA_VISIBLE_DEVICES'] = "3"


In [ ]:
# Constants.

DATA_CSV = Path("artifacts/processed_data/chexpertplus_frontal_5labels.csv")
LINEAR_PROBE_ROOT = Path("artifacts/probing/linear_probe")
RESULTS_DIR = LINEAR_PROBE_ROOT / "results"
PROBES_DIR = LINEAR_PROBE_ROOT / "probes"

METRICS_CSV = RESULTS_DIR / "experiment_metrics.csv"
YES_NO_CSV = RESULTS_DIR / "experiment_yes_no_scores.csv"

MEDSIGLIP_MODEL_ID = "google/medsiglip-448"
MEDGEMMA_MODEL_ID = "google/medgemma-4b-it"
MODEL_DTYPE = torch.bfloat16
FEATURE_DTYPE = np.float32

models = [
    {"name": "base_medgemma", "adapter_path": None},
    {"name": "lora_image_first", "adapter_path": Path("artifacts/lora_sft/image_first_adapter")},
    {"name": "lora_text_first", "adapter_path": Path("artifacts/lora_sft/text_first_adapter")},
]

IMAGE_LOAD_NUM_WORKERS = 4
MEDSIGLIP_BATCH_SIZE = 512
MEDGEMMA_BATCH_SIZE = 24
PROBE_PARALLEL_JOBS = 16
PROBE_INNER_NUM_THREADS = 8
MAX_ITER = 2000
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

TARGET_LABELS = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Pleural Effusion",
]

for folder in [RESULTS_DIR, PROBES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


In [ ]:
# Load the processed CheXpert+ manifest.

df = pd.read_csv(DATA_CSV)
labels = (df[TARGET_LABELS].to_numpy() == 1).astype(np.int8)
train_idx = np.where(df["probe_split"].to_numpy() == "train")[0]
test_idx = np.where(df["probe_split"].to_numpy() == "test")[0]
image_paths = df["image_path"].tolist()

print("rows", len(df))
print("train", len(train_idx), "test", len(test_idx))
display(pd.DataFrame({
    "label": TARGET_LABELS,
    "train_positive_rate": labels[train_idx].mean(axis=0),
    "test_positive_rate": labels[test_idx].mean(axis=0),
}))


In [ ]:
# Minimal helpers.

def slug(text):
    return re.sub(r"[^a-z0-9]+", "_", str(text).lower()).strip("_")


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def fit_probe(feature_matrix, layer, label, model_name, prompt_order, condition, feature_name):
    label_i = TARGET_LABELS.index(label)
    x = feature_matrix if layer is None else feature_matrix[:, layer, :]
    x_train = x[train_idx].astype(np.float32, copy=False)
    x_test = x[test_idx].astype(np.float32, copy=False)
    y_train = labels[train_idx, label_i]
    y_test = labels[test_idx, label_i]

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=1.0, solver="lbfgs", max_iter=MAX_ITER, random_state=RANDOM_STATE),
    )

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        with threadpool_limits(limits=PROBE_INNER_NUM_THREADS):
            model.fit(x_train, y_train)

    layer_name = "layer_none" if layer is None else f"layer_{int(layer):02d}"
    model_path = PROBES_DIR / slug(model_name) / slug(prompt_order) / slug(label) / layer_name / f"{slug(feature_name)}.pkl"
    model_path.parent.mkdir(parents=True, exist_ok=True)
    with model_path.open("wb") as f:
        pickle.dump(model, f)

    scores = model.predict_proba(x_test)[:, 1]
    return {
        "model_name": model_name,
        "prompt_order": prompt_order,
        "condition": condition,
        "feature": feature_name,
        "layer": layer,
        "label": label,
        "feature_dim": int(x_train.shape[1]),
        "positive_prevalence": float(y_test.mean()),
        "auroc": float(roc_auc_score(y_test, scores)),
        "auprc": float(average_precision_score(y_test, scores)),
        "n_iter": int(model.named_steps["logisticregression"].n_iter_[0]),
        "model_path": str(model_path),
    }


In [ ]:
# Cache images in CPU RAM once.

with ThreadPoolExecutor(max_workers=IMAGE_LOAD_NUM_WORKERS) as pool:
    images = list(tqdm(pool.map(load_rgb, image_paths), total=len(image_paths), desc="Load images"))

print("cached images", len(images))


In [ ]:
# Prepare output containers.

metric_rows = []
yes_no_rows = []


In [ ]:
# Extract standalone MedSigLIP features and train baseline probes.

device = "cuda" if torch.cuda.is_available() else "cpu"
medsiglip_processor = AutoProcessor.from_pretrained(MEDSIGLIP_MODEL_ID)
medsiglip_model = AutoModel.from_pretrained(MEDSIGLIP_MODEL_ID, dtype=MODEL_DTYPE).to(device).eval()
medsiglip_dim = medsiglip_model.config.vision_config.hidden_size
medsiglip_features = np.empty((len(df), medsiglip_dim), dtype=FEATURE_DTYPE)

for start in tqdm(range(0, len(images), MEDSIGLIP_BATCH_SIZE), desc="MedSigLIP"):
    end = min(start + MEDSIGLIP_BATCH_SIZE, len(images))
    inputs = medsiglip_processor(images=images[start:end], return_tensors="pt")
    inputs = {
        k: v.to(device=device, dtype=MODEL_DTYPE) if v.is_floating_point() else v.to(device=device)
        for k, v in inputs.items()
    }
    with torch.inference_mode():
        outputs = medsiglip_model.get_image_features(**inputs)
        emb = outputs.pooler_output if hasattr(outputs, "pooler_output") else outputs
        emb = emb / emb.norm(p=2, dim=-1, keepdim=True)
    medsiglip_features[start:end] = emb.float().cpu().numpy().astype(FEATURE_DTYPE)

jobs = [(None, label) for label in TARGET_LABELS]
metric_rows.extend(Parallel(n_jobs=PROBE_PARALLEL_JOBS, prefer="threads")(
    delayed(fit_probe)(medsiglip_features, layer, label, "medsiglip_standalone", "baseline", "medsiglip_standalone", "medsiglip_standalone")
    for layer, label in tqdm(jobs, desc="MedSigLIP probes")
))

del medsiglip_model, medsiglip_processor, medsiglip_features
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Load MedGemma processor and process images once with the MedGemma image processor.

medgemma_processor = AutoProcessor.from_pretrained(MEDGEMMA_MODEL_ID)

pixel_chunks = []
for start in tqdm(range(0, len(images), MEDGEMMA_BATCH_SIZE), desc="MedGemma image processor"):
    end = min(start + MEDGEMMA_BATCH_SIZE, len(images))
    pixel_inputs = medgemma_processor.image_processor(images=images[start:end], return_tensors="pt", do_pan_and_scan=False)
    pixel_chunks.append(pixel_inputs["pixel_values"].to(dtype=MODEL_DTYPE, device="cpu"))
medgemma_pixel_values = torch.cat(pixel_chunks, dim=0)
del pixel_chunks

print("cached pixel_values", tuple(medgemma_pixel_values.shape), medgemma_pixel_values.dtype)


In [ ]:
# Prompt/token helpers kept here because the image token expansion must match Gemma 3 processing.

def prompt_text(prompt_order, label):
    question = f"Question: Is there {label.lower()} in this image? Answer yes or no."
    image_item = {"type": "image"}
    if prompt_order == "image_first":
        content = [image_item, {"type": "text", "text": f"\n{question}\nAnswer: "}]
    else:
        content = [{"type": "text", "text": f"{question}\n"}, image_item, {"type": "text", "text": "\nAnswer: "}]
    messages = [{"role": "user", "content": content}]
    text = medgemma_processor.apply_chat_template(messages, add_generation_prompt=False, tokenize=False)
    if isinstance(text, list):
        text = text[0]
    return text.replace(medgemma_processor.boi_token, medgemma_processor.full_image_sequence)


def tokenize_prompt(prompt_order, label, batch_size):
    inputs = medgemma_processor.tokenizer(
        [prompt_text(prompt_order, label)] * batch_size,
        return_tensors="pt",
        padding=True,
    )

    if hasattr(medgemma_processor, "create_mm_token_type_ids"):
        token_type_ids = medgemma_processor.create_mm_token_type_ids(inputs["input_ids"])
        if not torch.is_tensor(token_type_ids):
            token_type_ids = torch.tensor(token_type_ids)
        inputs["token_type_ids"] = token_type_ids

    return inputs

YES_TOKEN_ID = medgemma_processor.tokenizer("yes", add_special_tokens=False).input_ids[0]
NO_TOKEN_ID = medgemma_processor.tokenizer("no", add_special_tokens=False).input_ids[0]
print("yes/no token ids", YES_TOKEN_ID, NO_TOKEN_ID)


In [ ]:
# Run each MedGemma model, train probes, and collect yes/no scores.

for model_info in models:
    model_name = model_info["name"]
    adapter_path = model_info["adapter_path"]
    print("model", model_name)

    medgemma_model = AutoModelForImageTextToText.from_pretrained(
        MEDGEMMA_MODEL_ID,
        dtype=MODEL_DTYPE,
        device_map="auto",
    ).eval()
    if adapter_path is not None:
        medgemma_model = PeftModel.from_pretrained(medgemma_model, str(adapter_path)).eval()

    model_device = next(medgemma_model.parameters()).device
    model_config = medgemma_model.config if hasattr(medgemma_model, "config") else medgemma_model.base_model.model.config
    num_layers = model_config.text_config.num_hidden_layers
    hidden_size = model_config.text_config.hidden_size
    vision_hidden_size = model_config.vision_config.hidden_size
    image_token_id = getattr(model_config, "image_token_id", None)
    if image_token_id is None:
        image_token_id = getattr(model_config, "image_token_index")

    pre_projector_cache = {}
    projector_norm = next(module for name, module in medgemma_model.named_modules() if name.endswith("multi_modal_projector.mm_soft_emb_norm"))
    projector_hook = projector_norm.register_forward_hook(lambda module, inputs, output: pre_projector_cache.__setitem__("x", inputs[0].detach()))

    check_inputs = tokenize_prompt("image_first", TARGET_LABELS[0], 1)
    print("num_layers", num_layers, "hidden_size", hidden_size, "vision_hidden_size", vision_hidden_size)
    print("image token count", int(check_inputs["input_ids"].eq(image_token_id).sum().item()))

    for prompt_order in ["image_first", "text_first"]:
        for label in TARGET_LABELS:
            print(model_name, prompt_order, label)

            preprojector_features = np.empty((len(df), vision_hidden_size), dtype=FEATURE_DTYPE)
            predecoder_features = np.empty((len(df), hidden_size), dtype=FEATURE_DTYPE)
            layer_mean_features = np.empty((len(df), num_layers, hidden_size), dtype=FEATURE_DTYPE)
            layer_last_features = np.empty((len(df), num_layers, hidden_size), dtype=FEATURE_DTYPE)
            final_prompt_features = np.empty((len(df), num_layers, hidden_size), dtype=FEATURE_DTYPE)
            yes_logprob = np.empty(len(df), dtype=np.float32)
            no_logprob = np.empty(len(df), dtype=np.float32)

            for start in tqdm(range(0, len(df), MEDGEMMA_BATCH_SIZE), desc=f"{model_name} {prompt_order} {label}"):
                end = min(start + MEDGEMMA_BATCH_SIZE, len(df))
                batch_size = end - start
                inputs = tokenize_prompt(prompt_order, label, batch_size)
                inputs["pixel_values"] = medgemma_pixel_values[start:end]
                inputs = {
                    k: v.to(device=model_device, dtype=MODEL_DTYPE) if v.is_floating_point() else v.to(device=model_device)
                    for k, v in inputs.items()
                }

                with torch.inference_mode():
                    outputs = medgemma_model(
                        **inputs,
                        output_hidden_states=True,
                        use_cache=False,
                        logits_to_keep=1,
                        return_dict=True,
                    )

                preprojector_features[start:end] = pre_projector_cache["x"].float().mean(dim=1).cpu().numpy().astype(FEATURE_DTYPE)
                predecoder_features[start:end] = outputs.image_hidden_states.float().mean(dim=1).cpu().numpy().astype(FEATURE_DTYPE)

                log_probs = torch.log_softmax(outputs.logits[:, -1, :].float(), dim=-1)
                yes_logprob[start:end] = log_probs[:, YES_TOKEN_ID].cpu().numpy()
                no_logprob[start:end] = log_probs[:, NO_TOKEN_ID].cpu().numpy()

                image_mask = inputs["input_ids"].eq(image_token_id)
                image_token_count = int(image_mask.sum(dim=1)[0].item())
                last_positions = inputs["attention_mask"].sum(dim=1) - 1
                batch_arange = torch.arange(batch_size, device=model_device)

                for layer_i, hidden in enumerate(outputs.hidden_states[1:]):
                    image_hidden = hidden[image_mask].reshape(batch_size, image_token_count, hidden_size).float()
                    layer_mean_features[start:end, layer_i] = image_hidden.mean(dim=1).cpu().numpy().astype(FEATURE_DTYPE)
                    layer_last_features[start:end, layer_i] = image_hidden[:, -1].cpu().numpy().astype(FEATURE_DTYPE)
                    final_prompt_features[start:end, layer_i] = hidden[batch_arange, last_positions].float().cpu().numpy().astype(FEATURE_DTYPE)

                del outputs, inputs

            metric_rows.append(
                fit_probe(preprojector_features, None, label, model_name, prompt_order, label, "medgemma_pre_projector_mean_image_token")
            )
            metric_rows.append(
                fit_probe(predecoder_features, None, label, model_name, prompt_order, label, "medgemma_pre_decoder_mean_image_token")
            )

            for feature_name, feature_matrix in [
                ("medgemma_layer_mean_image_token", layer_mean_features),
                ("medgemma_layer_last_image_token", layer_last_features),
            ]:
                jobs = [(layer, label) for layer in range(num_layers)]
                metric_rows.extend(Parallel(n_jobs=PROBE_PARALLEL_JOBS, prefer="threads")(
                    delayed(fit_probe)(feature_matrix, layer, target_label, model_name, prompt_order, label, feature_name)
                    for layer, target_label in tqdm(jobs, desc=f"{feature_name} probes")
                ))

            jobs = [(layer, label) for layer in range(num_layers)]
            metric_rows.extend(Parallel(n_jobs=PROBE_PARALLEL_JOBS, prefer="threads")(
                delayed(fit_probe)(final_prompt_features, layer, target_label, model_name, prompt_order, label, "medgemma_layer_final_prompt_token")
                for layer, target_label in tqdm(jobs, desc="final prompt token probes")
            ))

            label_i = TARGET_LABELS.index(label)
            score = yes_logprob - no_logprob
            yes_no_rows.append(pd.DataFrame({
                "model_name": model_name,
                "study_id": df["study_id"],
                "subject_id": df["subject_id"],
                "dicom_id": df["dicom_id"],
                "probe_split": df["probe_split"],
                "prompt_order": prompt_order,
                "label": label,
                "y_true": labels[:, label_i],
                "logprob_yes": yes_logprob,
                "logprob_no": no_logprob,
                "score_yes_minus_no": score,
                "pred_yes": score > 0,
            }))
            metric_rows.append({
                "model_name": model_name,
                "prompt_order": prompt_order,
                "condition": label,
                "feature": "medgemma_yes_no_logprob",
                "layer": None,
                "label": label,
                "feature_dim": 1,
                "positive_prevalence": float(labels[test_idx, label_i].mean()),
                "auroc": float(roc_auc_score(labels[test_idx, label_i], score[test_idx])),
                "auprc": float(average_precision_score(labels[test_idx, label_i], score[test_idx])),
                "n_iter": None,
                "model_path": "",
            })

            del preprojector_features, predecoder_features, layer_mean_features, layer_last_features, final_prompt_features, yes_logprob, no_logprob
            gc.collect()
            torch.cuda.empty_cache()

    projector_hook.remove()
    del medgemma_model
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# Save results.

metrics_df = pd.DataFrame(metric_rows)
metrics_df["layer"] = pd.to_numeric(metrics_df["layer"], errors="coerce")
metrics_df = metrics_df.sort_values(["model_name", "prompt_order", "condition", "feature", "layer", "label"], na_position="first")
yes_no_df = pd.concat(yes_no_rows, ignore_index=True)

metrics_df.to_csv(METRICS_CSV, index=False)
yes_no_df.to_csv(YES_NO_CSV, index=False)

print("metrics rows", len(metrics_df))
print("yes/no rows", len(yes_no_df))
print("saved", METRICS_CSV)
print("saved", YES_NO_CSV)
print("saved probes to", PROBES_DIR)

display(metrics_df)
display(
    metrics_df
    .groupby(["model_name", "prompt_order", "feature", "layer"], dropna=False, as_index=False)
    .agg(mean_auroc=("auroc", "mean"), mean_auprc=("auprc", "mean"))
    .sort_values("mean_auroc", ascending=False)
)
